## Tools

#### models can request to call the tools to perform tasks like fetching data from database, searching web, or running code.

##### 1.A schema, including the name of the tool, a description, and/or argument definitions(often a JSON schema)
##### 2.A function or coroutine to execute

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("why do parrots speak?")
response.content

'<think>\nOkay, so the user is asking why parrots can speak. Let me start by recalling what I know about parrots and their vocal abilities.\n\nFirst, I remember that parrots are part of the Psittaciformes order. They\'re known for their ability to mimic human speech, right? But why exactly can they do that? I think it has something to do with their vocal organs and brain structure. Let me break it down.\n\nParrots have a unique vocal organ called the syrinx. I think that\'s the part of the bird\'s respiratory system where sound is produced, similar to the human larynx. But how is it different? Maybe the syrinx in parrots allows them to produce a wider range of sounds because they can manipulate it in different ways. Humans use their vocal cords, but parrots might have more control over their syrinx, enabling them to mimic sounds better.\n\nThen there\'s the brain aspect. I\'ve heard that parrots have a part of the brain called the nidopallium, which is involved in learning and processi

In [5]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """ Get weather at a location """
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [7]:
response = model_with_tools.invoke("what's the weather in chennai?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tools:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Chennai. I need to use the get_weather function. The function requires a location parameter. Chennai is a city in India, so I should specify that as the location. I\'ll call the function with "Chennai, India" to ensure it\'s clear. Let me format the tool call correctly.\n', 'tool_calls': [{'id': 'gn8wf8rjz', 'function': {'arguments': '{"location":"Chennai, India"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 153, 'total_tokens': 250, 'completion_time': 0.164696554, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.006057023, 'prompt_tokens_details': None, 'queue_time': 0.056594697, 'total_time': 0.170753577}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run

## Tool Execution Loops

In [8]:
#step 1: model generates tool calls
messages = [{"role":"user","content":"what is the weather in chennai?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#step 2: execute the tool and collect the results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Chennai is sunny.


In [9]:
messages

[{'role': 'user', 'content': 'what is the weather in chennai?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Chennai. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Chennai, I need to call this function with "Chennai" as the location. I\'ll make sure the arguments are correctly formatted in JSON within the tool_call tags.\n', 'tool_calls': [{'id': '3kf2zctfp', 'function': {'arguments': '{"location":"Chennai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 153, 'total_tokens': 248, 'completion_time': 0.142440389, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.006275743, 'prompt_tokens_details': None, 'queue_time': 0.159894785, 'total_time': 0.148716132}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier':